# SEED-0002 — R3 final-style vLLM SC16 vs explicit 16-seed SC16 (dev only)

Directly compares the current final-style `SamplingParams(n=16, seed=3)` setup to the completed explicit-unique-16-seed pool. Prompt, R3 adapter, temperature, top-p, output limit, parser, vote, and held dev split are fixed. PAL and adaptive length are excluded to isolate sampling policy.

In [ ]:
# Cell 1 — Fresh A100 only. Run once, then restart the runtime before Cell 2.
%pip install -q --no-cache-dir "nvidia-cuda-runtime==13.0.88" "nvidia-cuda-nvrtc==13.0.88" "vllm==0.26.0" "pandas>=2.2,<3"
print("[SETUP] Restart the runtime once, then run Cells 2–5 in order.")


In [ ]:
# Cell 2 — Mount Drive, update the private repo, and cache the pinned base model.
from google.colab import drive, userdata
from pathlib import Path
import subprocess

drive.mount('/content/drive')
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = 'https://github.com/jhparktime/qwen-math-final-2026.git'
if token:
    url = 'https://x-access-token:' + token + '@github.com/jhparktime/qwen-math-final-2026.git'
repo = Path('/content/qwen-math-final')
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', url, str(repo)], check=True)
%cd /content/qwen-math-final
!python3 scripts/prefetch_model.py


In [ ]:
# Cell 3 — Freeze inputs: the same dev split, R3 adapter, and completed explicit16 pool.
import re, unicodedata
from pathlib import Path

def compact(value):
    return re.sub(r'[\s_-]+', '', unicodedata.normalize('NFC', str(value)).casefold())

roots = [p for p in Path('/content/drive/MyDrive').iterdir() if p.is_dir() and compact(p.name) == compact('2026소중한챌린지')]
assert len(roots) == 1, roots
project = roots[0]
runs = project / 'runs'
DEV_PATH = runs / 'AUDIT-0002-clean-split-passN-20260821-215706' / 'splits' / 'dev_v1.csv'
R3_ADAPTER = runs / 'RFT-0008D-r3mix-r2continue-r16-a100' / 'adapter_final'
EXPLICIT_RAW = runs / 'SEED-0001-r3-explicit-seedcount-dev' / 'candidates' / 'dev_r3_explicit16_seed3_pool.jsonl'
OUTPUT_DIR = runs / 'SEED-0002-r3-vllmn16-vs-explicit16-dev'
for path in [DEV_PATH, R3_ADAPTER / 'adapter_config.json', R3_ADAPTER / 'adapter_model.safetensors', EXPLICIT_RAW]:
    assert path.exists(), path
print({'dev': str(DEV_PATH), 'adapter': str(R3_ADAPTER), 'explicit_pool': str(EXPLICIT_RAW), 'base_seed': 3, 'output': str(OUTPUT_DIR)})


In [ ]:
# Cell 4 — Generate only the missing final-style vLLM n=16 candidates, then compare against explicit16.
import os, subprocess, sys
env = dict(os.environ, PYTHONPATH='.')
subprocess.run([sys.executable, 'inference/compare_vllm_n16_explicit_dev.py', '--input', str(DEV_PATH), '--adapter', str(R3_ADAPTER), '--explicit-raw', str(EXPLICIT_RAW), '--output-dir', str(OUTPUT_DIR), '--base-seed', '3'], check=True, env=env)


In [ ]:
# Cell 5 — Read the direct comparison and keep the report before changing any final inference policy.
import json
report = json.loads((OUTPUT_DIR / 'reports' / 'experiment_report.json').read_text())
print(json.dumps({key: report[key] for key in ['final_style_vllm_n16', 'explicit16_unique_seed', 'delta_explicit_minus_vllm', 'changed_rows', 'fixes', 'breaks']}, ensure_ascii=False, indent=2))


In [ ]:
# Final cell — optional GPU release after all artifacts are saved.
DISCONNECT_GPU_RUNTIME = False
if DISCONNECT_GPU_RUNTIME:
    from google.colab import runtime
    runtime.unassign()
else:
    print('[RUNTIME] retained. Set DISCONNECT_GPU_RUNTIME=True when finished.')
